# Classificatore spam/ham con TF-IDF e TensorFlow

Pipeline: CSV → split train/validation/test → `TextVectorization` (output `tf_idf`) → rete densa con output sigmoide.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
tf.keras.utils.set_random_seed(SEED)

In [2]:
df = pd.read_csv("../datasets/italian.csv")  # colonne: text, labels (1 = spam, 0 = ham)
print("Righe nel file:", len(df))

# Il dataset contiene messaggi ripetuti: se lo stesso testo finisse sia nel train sia nel test,
# le metriche sul test sarebbero gonfiate (il modello l'avrebbe già visto). Teniamo una sola copia di ogni testo.
df = df.drop_duplicates(subset="text").reset_index(drop=True)
df["label"] = df["labels"].astype(int)
print("Righe dopo la rimozione dei duplicati:", len(df))
print(df["label"].map({1: "spam", 0: "ham"}).value_counts(normalize=True))
df.head()

Righe nel file: 18719
Righe dopo la rimozione dei duplicati: 15144
label
spam    0.52047
ham     0.47953
Name: proportion, dtype: float64


,text,labels,label
0,Offerta limitata! Guadagna 5000 euro al giorno...,1,1
1,Attenzione! Il tuo account è stato selezionato...,1,1
2,Pubblicità commerciale non richiesta: Scopri i...,1,1
3,"Ciao, ti ricordo l'appuntamento di domani alle...",0,0
4,Congratulazioni! Hai vinto un voucher da 500 e...,1,1


In [3]:
# Split stratificato: così da mantenere la stessa proporzione di messaggi spam e non spam in ogni insieme
X, y = df["text"].tolist(), df["label"].to_numpy()

# 1) il test (20%) viene messo da parte e usato solo alla fine
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)
# 2) dal resto si ricava la validation (10% del train+val), usata per early stopping e per scegliere gli iperparametri
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1, stratify=y_trainval, random_state=SEED,
)
# Keras 3 non accetta array numpy di stringhe: li passiamo come tensori tf.string
X_train, X_val, X_test = tf.constant(X_train), tf.constant(X_val), tf.constant(X_test)

## Stop words:
In genere sui classificatori TF-IDF si tolgono per migliorare il modello. Qui non lo facciamo: l'IDF smorza già il peso delle parole molto comuni, e alcune di esse ("tuo", "ora", "subito") possono essere indicative di spam.

In [4]:
print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {X_test.shape}")
print(X_train[:5]) # sample of the first 5 messages in the training set
# conta il numero di parole distinte nel train set
vocab = set()
for message in X_train.numpy():
    vocab.update(message.decode("utf-8").split())
print(f"Number of distinct words in the training set: {len(vocab)}")


Train shape: (10903,), Validation shape: (1212,), Test shape: (3029,)
tf.Tensor(
[b'La temperatura prevista per domani \xc3\xa8 di 22 gradi con cielo sereno.'
 b'Tutti i medici ti consigliano di prendere questo integratore! \xf0\x9f\x8c\xbf'
 b'Puoi ritirare subito i tuoi soldi: Vandiamo il tuo credito'
 b'Urgent! Your account will be locked in 24h. Confirm your details by clicking this link.'
 b'Hai vinto un viaggio pagato da 5 giorni a Dubai! Rispondi a questa email per confermare la tua vincita.'], shape=(5,), dtype=string)
Number of distinct words in the training set: 19088


In [5]:
# dimensioanmento del vocabolario con il tokenizer di Keras: per evitare overfitting, 
# si può limitare il numero di parole distinte da considerare, prendendo quelle che 
# compaiono almeno 2 volte
counter = tf.keras.layers.TextVectorization(
    output_mode="count",
    standardize="lower_and_strip_punctuation"
)
counter.adapt(X_train)
counts = counter(X_train).numpy().sum(axis=0)
print(f"Parole distinte (tokenizer Keras): {len(counts)}")
print(f"Parole con almeno 2 occorrenze: {(counts >= 2).sum()}")

# quindi, approssimiamo per eccesso alla centinaia più vicina, così da avere un numero di parole distinte che sia multiplo di 100
# il numero di parole distinte da considerare, prendendo quelle che compaiono almeno 2 volte
counts = counts[counts >= 2]
vocab_size = int(np.ceil(len(counts) / 100) * 100)
print(f"Vocab size (words with at least 2 occurrences, rounded up to nearest 100): {vocab_size}")

Parole distinte (tokenizer Keras): 11145
Parole con almeno 2 occorrenze: 4972
Vocab size (words with at least 2 occurrences, rounded up to nearest 100): 5000


## Vettorizzazione TF-IDF

Il layer `TextVectorization` con `output_mode="tf_idf"` impara vocabolario e pesi IDF dal solo training set (`adapt`), escluse validation e test, così nessuna informazione di questi insiemi entra nel vettorizzatore.

In [6]:
MAX_TOKENS = vocab_size

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="tf_idf",
    standardize="lower_and_strip_punctuation",
)
vectorizer.adapt(X_train)
print(vectorizer.get_vocabulary()[:20])

['[UNK]', np.str_('il'), np.str_('per'), np.str_('di'), np.str_('la'), np.str_('un'), np.str_('e'), np.str_('in'), np.str_('al'), np.str_('a'), np.str_('è'), np.str_('con'), np.str_('tuo'), np.str_('i'), np.str_('qui'), np.str_('ho'), np.str_('una'), np.str_('clicca'), np.str_('ora'), np.str_('subito')]


In [7]:
# vediamo come sono rappresentati i primi 5 messaggi del train set con il TF-IDF (solo le posizioni diverse da zero)
vocab_list = [str(w) for w in vectorizer.get_vocabulary()]
vectors = vectorizer(X_train[:5]).numpy()

for text, vec in zip(X_train[:5].numpy(), vectors):
    nonzero = np.nonzero(vec)[0]
    print(text.decode("utf-8"))
    print(f"  vettore: {vec.shape[0]} posizioni, {len(nonzero)} diverse da zero")
    for i in sorted(nonzero, key=lambda i: -vec[i]):
        print(f"    [{i:>4}] {vocab_list[i]:<12} {vec[i]:.2f}")
    print()

La temperatura prevista per domani è di 22 gradi con cielo sereno.
  vettore: 5000 posizioni, 12 diverse da zero
    [1508] 22           6.81
    [1282] sereno       6.66
    [1203] temperatura  6.59
    [1220] prevista     6.59
    [ 702] gradi        5.90
    [ 680] cielo        5.87
    [  40] domani       3.11
    [  11] con          2.19
    [  10] è            2.16
    [   4] la           1.86
    [   3] di           1.34
    [   2] per          1.26

Tutti i medici ti consigliano di prendere questo integratore! 🌿
  vettore: 5000 posizioni, 9 diverse da zero
    [   0] [UNK]        14.35
    [1458] medici       6.81
    [ 696] prendere     5.90
    [ 127] integratore  4.18
    [  65] tutti        3.63
    [  37] questo       3.03
    [  26] ti           2.85
    [  13] i            2.29
    [   3] di           1.34

Puoi ritirare subito i tuoi soldi: Vandiamo il tuo credito
  vettore: 5000 posizioni, 10 diverse da zero
    [   0] [UNK]        7.18
    [ 367] credito      5.18
   

## Modello

In [8]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 1, 5000)        │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1, 64)          │       320,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1, 1)           │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,129 (1.22 MB)

 Trainable params: 320,129 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
)

Epoch 1/10


  1/341 ━━━━━━━━━━━━━━━━━━━━ 1:22 243ms/step - accuracy: 0.4062 - loss: 0.7159

 49/341 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7991 - loss: 0.4744    

108/341 ━━━━━━━━━━━━━━━━━━━━ 0s 939us/step - accuracy: 0.8744 - loss: 0.3198

171/341 ━━━━━━━━━━━━━━━━━━━━ 0s 888us/step - accuracy: 0.9037 - loss: 0.2519

...

182/341 ━━━━━━━━━━━━━━━━━━━━ 0s 844us/step - accuracy: 0.9857 - loss: 0.0416

239/341 ━━━━━━━━━━━━━━━━━━━━ 0s 854us/step - accuracy: 0.9859 - loss: 0.0404

270/341 ━━━━━━━━━━━━━━━━━━━━ 0s 946us/step - accuracy: 0.9863 - loss: 0.0392

327/341 ━━━━━━━━━━━━━━━━━━━━ 0s 935us/step - accuracy: 0.9868 - loss: 0.0381

341/341 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9870 - loss: 0.0376 - val_accuracy: 0.9546 - val_loss: 0.1244


## Valutazione

In [10]:
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))
print(confusion_matrix(y_test, y_pred))

 1/95 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step

28/95 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


              precision    recall  f1-score   support

         ham       0.96      0.96      0.96      1452
        spam       0.96      0.96      0.96      1577

    accuracy                           0.96      3029
   macro avg       0.96      0.96      0.96      3029
weighted avg       0.96      0.96      0.96      3029

[[1393   59]
 [  57 1520]]


In [11]:
def predict(texts):
    probs = model.predict(tf.constant(texts), verbose=0).ravel()
    return [("spam" if p > 0.5 else "ham", float(p)) for p in probs]

predict([
    "Complimenti! Hai vinto un premio, clicca subito sul link per riscuoterlo",
    "Ciao, ci vediamo stasera a cena?",
])

[('spam', 0.995888888835907), ('ham', 0.016587888821959496)]

## Versione minima: regressione logistica

Solo vettorizzatore + un neurone sigmoide, senza layer nascosto né dropout. Il vettorizzatore è già adattato sul train, quindi viene riusato così com'è.

In [12]:
logreg = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
logreg.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

logreg.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=0,
)

y_pred_lr = (logreg.predict(X_test).ravel() > 0.5).astype(int)
print(classification_report(y_test, y_pred_lr, target_names=["ham", "spam"]))
print(confusion_matrix(y_test, y_pred_lr))

 1/95 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step

95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 515us/step


              precision    recall  f1-score   support

         ham       0.96      0.96      0.96      1452
        spam       0.97      0.97      0.97      1577

    accuracy                           0.97      3029
   macro avg       0.96      0.96      0.96      3029
weighted avg       0.97      0.97      0.97      3029

[[1400   52]
 [  54 1523]]


In [13]:
# Ogni parola ha un peso: positivo spinge verso spam, negativo verso ham
weights = logreg.layers[-1].get_weights()[0].ravel()
vocab_list = vectorizer.get_vocabulary()
order = np.argsort(weights)
print("Parole più indicative di spam:", [(vocab_list[i], round(float(weights[i]), 2)) for i in order[::-1][:10]])
print("Parole più indicative di ham: ", [(vocab_list[i], round(float(weights[i]), 2)) for i in order[:10]])

Parole più indicative di spam: [(np.str_('clicca'), 0.44), (np.str_('qui'), 0.42), (np.str_('ora'), 0.39), (np.str_('offerta'), 0.39), (np.str_('subito'), 0.38), (np.str_('guadagna'), 0.37), (np.str_('adulti'), 0.34), (np.str_('investi'), 0.34), (np.str_('segreto'), 0.33), (np.str_('malware'), 0.33)]
Parole più indicative di ham:  [(np.str_('ho'), -0.38), (np.str_('progetto'), -0.33), (np.str_('ricordo'), -0.32), (np.str_('grazie'), -0.3), (np.str_('caffè'), -0.29), (np.str_('pomeriggio'), -0.28), (np.str_('consigli'), -0.28), (np.str_('stasera'), -0.28), (np.str_('ieri'), -0.27), (np.str_('giornata'), -0.27)]


In [14]:
for i in ["liao", "he","said","tot","reach", "i", "u", "[UNK]"]:
    idx = vocab_list.index(i) if i in vocab_list else None
    if idx is not None:
        print(f"'{i}': weight={weights[idx]:.3f}")
        
    else:
        print(f"'{i}': not in vocabulary")

'liao': not in vocabulary
'he': not in vocabulary
'said': not in vocabulary
'tot': not in vocabulary
'reach': not in vocabulary
'i': weight=-0.008
'u': not in vocabulary
'[UNK]': weight=0.004


In [15]:
# Come la regressione logistica classifica il primo messaggio del train: p(spam) = sigmoide(somma di peso_parola × valore_tfidf + bias)
w = logreg.layers[-1].get_weights()[0].ravel()
b = float(logreg.layers[-1].get_weights()[1][0])

x = vectorizer(X_train[:1]).numpy()[0]
nonzero = np.nonzero(x)[0]
print(X_train[0].numpy().decode("utf-8"))
print(f"{'parola':<10} {'tf-idf':>7} {'peso':>7} {'contributo':>11}")
for i in sorted(nonzero, key=lambda i: -abs(w[i] * x[i])):
    print(f"{str(vocab_list[i]):<10} {x[i]:>7.3f} {w[i]:>7.3f} {w[i] * x[i]:>11.3f}")

z = float(np.dot(w, x) + b)
print(f"\nbias = {b:.3f}")
print(f"z = somma dei contributi + bias = {z:.3f}")
print(f"p(spam) = sigmoide(z) = {1 / (1 + np.exp(-z)):.4f}")
print(f"logreg.predict         = {float(logreg.predict(X_train[:1], verbose=0)[0, 0]):.4f}")
print("classe:", "spam" if z > 0 else "ham")

La temperatura prevista per domani è di 22 gradi con cielo sereno.
parola      tf-idf    peso  contributo
cielo        5.866  -0.158      -0.925
22           6.813  -0.132      -0.901
domani       3.114  -0.241      -0.749
prevista     6.590  -0.110      -0.728
temperatura   6.590  -0.105      -0.692
sereno       6.659  -0.104      -0.690
è            2.160  -0.219      -0.474
gradi        5.898  -0.073      -0.432
la           1.855  -0.181      -0.336
di           1.338  -0.179      -0.239
per          1.259   0.077       0.096
con          2.191  -0.019      -0.043

bias = -0.153
z = somma dei contributi + bias = -6.266
p(spam) = sigmoide(z) = 0.0019
logreg.predict         = 0.0019
classe: ham
